In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def npg_palette():
    palette = ["#E64B35FF", "#4DBBD5FF", "#00A087FF", "#3C5488FF", "#F39B7FFF",
               "#8491B4FF", "#91D1C2FF", "#DC0000FF", "#7E6148FF", "#B09C85FF"]
    return sns.color_palette(palette, len(palette))

dpi = 300
npg = npg_palette()
npg


In [ ]:
dataname = "Nurses_local_int"
clean_name = "Nurses (local intercept)"

# dataname = "Nurses"
# clean_name = "Nurses (global intercept)"

# dataname = "trauma"
# clean_name = "Trauma Heterogeneous"

# dataname = "trauma_shuffled"
# clean_name = "Trauma Homogeneous"

df_all = pd.read_csv(f"data/summarized/params.{dataname}.csv", index_col=0)
df_all["Covariate"] = df_all["Covariate"].replace("sigma2", "$\\sigma^2$")
df_all["Method"] = df_all["Method"].str.replace(r"PDA.*", "ODAL" if "trauma" in dataname else "DLMM", regex=True)

df_intercept = df_all[df_all["Covariate"].str.startswith("Intercept")]
df = df_all.loc[~df_all.index.isin(df_intercept.index)]

colors = dict(zip(df_all["Method"].unique(), npg))

method_order = list(df_all["Method"].drop_duplicates().values)
method_order

In [ ]:
import math

n_hues = len(set(df["Method"]))
n_covs = len(set(df["Covariate"]))

def get_grid_size(n, k=0, fill=False):
    """
    Given interger n, find grid of size l*m = n that comes closest to being a square
    For k > 0,  returns successively "less square" grids
    Retruns (l, m)
    Can be used e.g. for plotting grids
    If fill, prevents pairs of form (1, m) by adding +1 to n
    """
    if n == 1: return (1, 1)
    from sympy import isprime
    if fill and isprime(n): n += 1
    pairs = []
    for i in reversed(range(1, n)):
        rows, cols = 0, 0
        if n % i == 0:
            cols = i
            rows = n // i
            pairs.append((rows, cols))

    pairs = [tuple(sorted(p)) for p in pairs]
    pairs = list(set(pairs))
    pairs = sorted(pairs, key=lambda x: np.abs(x[0] - x[1]))
    try:
        return pairs[k]
    except IndexError:
        return pairs[0]

rows, cols = get_grid_size(n_covs, fill=True)
n_covs, rows, cols

## Errorbars

In [ ]:
import matplotlib
from matplotlib.ticker import MaxNLocator

matplotlib.rcParams.update({'errorbar.capsize': 2})

# Calculate asymmetric error bars
df['err_low'] = df['Estimate'] - df['lower']
df['err_up'] = df['upper'] - df['Estimate']

grey = "#3A3A3A"

if rows*cols == len(df["Covariate"].unique()):
    covariates = df["Covariate"].unique().reshape(rows, cols)
else:
    covariates = df["Covariate"].unique()
    covariates = np.array(list(covariates) + [None])
    covariates = covariates.reshape(rows, cols)

fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), sharex=True)

for row in range(rows):
    for col in range(cols):

        covariate = covariates[row][col]

        ax = axes[row][col]
        if covariate is None: 
            ax.axis("off")
            continue

        dff = df[df["Covariate"]==covariate]

        sns.set_theme(style="ticks")
        sns.despine()
        #sns.despine(left=True, bottom=True)

        # Plot the points
        sns.stripplot(y='Estimate', x='Method', data=dff, hue="Method", size=8, palette=npg[:n_hues], ax=ax)

        try:
            ax.errorbar(
                y=dff['Estimate'], x=dff['Method'],
                yerr=[dff['err_low'], dff['err_up']], capsize=20, markeredgewidth=10,
                fmt='none', ecolor=colors.values(), linewidth=1.5
            )
        except ValueError:
            # weird error with colors, ignore. but cannot render errobrar caps
            pass

        ref = float(dff[dff["Method"]=="Combined"].loc[:, "Estimate"].values[0])
        ax.axhline(ref, ls="--", color=colors["Combined"])
        ax.set(xlabel=None)
        ax.set_title(covariate, color=grey)
        ax.set_ylabel("Estimate ±95% CI", color=grey)
        
        if row == rows - 1:
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha='right')
        else:
            ax.xaxis.set_ticks_position('none') 


        ax.tick_params(color=grey, labelcolor=grey)
        for spine in ax.spines.values():
            spine.set_edgecolor(grey)

        ax.set_xlim(ax.get_xlim())  # why does this add h padding? anyway, it works

        # micro tweaks
        if clean_name == "Nurses" and covariate == "age":
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

fig.suptitle(clean_name)
fig.tight_layout(w_pad=2, h_pad=3)
#fig.savefig(f"figures/errorbars.{dataname}.png", dpi=dpi)

## Barplots

In [ ]:
if "Truth_Estimate" not in df.columns:
    df["CI_Size"] = df["upper"] - df["lower"]
    truth = df[df["Method"] == "Combined"][["Covariate", "Estimate"]].rename(columns={"Estimate": "Truth_Estimate"})
    df = df.merge(truth, on="Covariate", how="left")
    df["Error"] = (df["Estimate"] - df["Truth_Estimate"]).abs()
    df["Error_signed"] = (df["Estimate"] - df["Truth_Estimate"])


fig, ax = plt.subplots(2, 1, figsize=(7, 7))
sns.despine()

sns.barplot(data=df[df["Method"]!="Combined"], x="Covariate", y="Error", hue="Method", palette=colors, ax=ax[0])
ax[0].set_ylabel("Absolute Error")

sns.barplot(data=df, x="Covariate", y="CI_Size", hue="Method", palette=colors, ax=ax[1])
ax[1].set_ylabel("CI Width")

handles, labels = ax[1].get_legend_handles_labels()
ax[0].legend(handles, labels, ncols=2)
ax[1].legend([], [], framealpha=0)
#ax[1].legend(title=None, ncols=2)

for a in ax:
    a.set_xlabel(None)

fig.suptitle(clean_name)
fig.tight_layout()
#fig.savefig(f"figures/bars.{dataname}.png", dpi=dpi)

## LaTeX table

In [ ]:
df_clean = df.rename({"lower": "Lower",
                      "upper": "Upper",
                      "Covariate": "Parameter"
                      }, axis=1)

df_clean = df_clean[["Parameter", "Method", "Estimate", "Lower", "Upper", "Error"]]
df_clean["Method"] = pd.Categorical(df_clean["Method"], categories=method_order, ordered=True)
df_clean = df_clean.sort_values(by=["Parameter", "Method"])
print(df_clean.to_latex(index=False))

## Composite figures

In [ ]:
def lolipop(data, ax, method_order, covs = None):

    if (covs is None):
        covs = data.groupby("Covariate").groups.keys()
    print(covs)
    pad = 0.1
    offset = (1 - 2*pad) / (len(data["Method"].unique()) - 1)
    x = 0


    for cov in covs:
        i = 0
        for meth in method_order:
            dd = data[(data["Method"]==meth) & (data["Covariate"]==cov)]
            if len(dd) != 1: 
                continue
            xx = -0.5 + pad + x + offset * i
            yy = dd["Error_signed"].iloc[0]
            ax.scatter(xx, yy, color=colors[meth], label=meth if x==0 else None)
            ax.plot((xx, xx), (0, yy), color=colors[meth])
            i += 1
        x += 1
        ax.axvline(x-0.5, ls="--", color="grey")

    ax.axhline(0, color="darkgrey", zorder=0)
    ylim = 1.05*max(np.abs(ax.get_ylim()[0]), np.abs(ax.get_ylim()[1]))
    ax.set_ylim(-ylim, ylim)
    ax.set_xlim(-0.5, len(covs)-0.5)
    ax.set_xticks(range(len(covs)))
    ax.set_xticklabels(covs)
    ax.set(ylabel="Error")
    ax.legend()
    
use_lolipop = True

### Nurses local intercept

In [ ]:
if clean_name.startswith("Nurses (local"):

    import matplotlib
    from matplotlib.ticker import MaxNLocator

    matplotlib.rcParams.update({'errorbar.capsize': 2})

    # Calculate asymmetric error bars
    df['err_low'] = df['Estimate'] - df['lower']
    df['err_up'] = df['upper'] - df['Estimate']

    grey = "#3A3A3A"
    df["Covariate"] = df["Covariate"].astype(str)

    if rows*cols == len(df["Covariate"].unique()):
        covariates = df["Covariate"].unique().reshape(rows, cols)
    else:
        covariates = df["Covariate"].unique()
        covariates = np.array(list(covariates) + [None])
        covariates = covariates.reshape(rows, cols)

    covariates_set = list(reversed(sorted(list({x for x in covariates.flatten() if x is not None}))))
    df["Covariate"] = pd.Categorical(df["Covariate"], categories=covariates_set, ordered=True)

    fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=False,
                            gridspec_kw={'width_ratios': [1, 1, 2], 'hspace': 0.27,'wspace':0.25})

    for row in range(rows):
        for col in range(cols):

            covariate = covariates[row][col]

            ax = axes[row][col]
            if covariate is None: 
                ax.axis("off")
                continue

            dff = df[df["Covariate"]==covariate]

            sns.set_theme(style="ticks")
            sns.despine()
            #sns.despine(left=True, bottom=True)

            # Plot the points
            sns.stripplot(y='Estimate', x='Method', data=dff, hue="Method", size=8, palette=npg[:n_hues], ax=ax)

            try:
                ax.errorbar(
                    y=dff['Estimate'], x=dff['Method'],
                    yerr=[dff['err_low'], dff['err_up']], capsize=20, markeredgewidth=10,
                    fmt='none', ecolor=colors.values(), linewidth=1.5
                )
            except ValueError:
                # weird error with colors, ignore. but cannot render errobrar caps
                pass

            ref = float(dff[dff["Method"]=="Combined"].loc[:, "Estimate"].values[0])
            ax.axhline(ref, ls="--", color=colors["Combined"])
            ax.set(xlabel=None)
            ax.set_title(covariate, color=grey)
            ax.set_ylabel("Estimate ±95% CI", color=grey)
            
            if row == rows - 1:
                ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha='right')
            else:
                ax.xaxis.set_ticks([])


            ax.tick_params(color=grey, labelcolor=grey)
            for spine in ax.spines.values():
                spine.set_edgecolor(grey)

            xlim = ax.get_xlim()
            offset = 0.1*(xlim[1]-xlim[0])
            ax.set_xlim(xlim[0]-offset, xlim[1]+offset)

            # micro tweaks
            if clean_name == "Nurses" and covariate == "age":
                ax.yaxis.set_major_locator(MaxNLocator(nbins=5))



    axes2 = axes[:, 2]
    if use_lolipop:
        lolipop(data=df[df["Method"]!="Combined"], ax=axes2[0], method_order=method_order, covs=covariates_set)
    else:
        sns.barplot(data=df[df["Method"]!="Combined"], x="Covariate", y="Error", hue="Method", palette=colors, ax=axes2[0])
        axes2[0].set_ylabel("Absolute Error")

    sns.barplot(data=df, x="Covariate", y="CI_Size", hue="Method", palette=colors, ax=axes2[1])
    axes2[1].set_ylabel("CI Width")
    axes2[1].set_ylim(0, 1.05*np.max(df["CI_Size"]))

    handles, labels = axes2[1].get_legend_handles_labels()
    axes2[1].legend(handles, labels, ncols=2)
    #axes2[0].legend([], [], framealpha=0)
    axes2[0].legend(ncols=1)
    if use_lolipop:
        axes2[0].legend(ncols=1, bbox_to_anchor=(0.375, 1.05), loc="upper center")
    for a in axes2:
        a.set_xlabel(None)

    flataxes = axes.flatten()
    # for i in range(len(flataxes)):
    #     flataxes[i].annotate(chr(ord('A')+i), xy=(-0.04, 1.1),
    #                xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[0].annotate("A", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[1].annotate("B", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[2].annotate("E", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[3].annotate("C", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[4].annotate("D", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[5].annotate("F", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
                        
    axes[-1][-1].plot(1,1)
    #fig.suptitle(clean_name)
    fig.tight_layout(w_pad=2, h_pad=2)
    fig.savefig(f"figures/collage.{dataname}.pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(f"figures/collage.{dataname}.png", dpi=dpi, bbox_inches="tight")

### Nurses global intercept

In [ ]:
if clean_name.startswith("Nurses (global"):

    from matplotlib.gridspec import GridSpec

    covariates_ordered = ['(Intercept)', 'age', 'experience', 'gender', 'wardtype', '$\\sigma^2$']
    df["Covariate"] = pd.Categorical(df["Covariate"], categories=covariates_ordered, ordered=True)

    scale = 1.4
    fig = plt.figure(figsize=(scale*12, scale*6))

    # Create a GridSpec with 2 rows and 6 columns (LCM of number of segments)
    gs = GridSpec(2, 12, height_ratios=[1, 1])  # Equal row height for now

    # --- Row 1
    ax1 = fig.add_subplot(gs[0, 0:2])           # First column (1 unit)
    ax2 = fig.add_subplot(gs[0, 2:4])           # Second column (1 unit)
    ax3 = fig.add_subplot(gs[0, 4:6])           # Third column (1 unit)
    ax4 = fig.add_subplot(gs[0, 6:])         # Fourth column (3 units)

    # --- Row 2
    ax5 = fig.add_subplot(gs[1, 0:2])           # First column (1 unit)
    ax6 = fig.add_subplot(gs[1, 2:4])           # Second column (1 unit)
    ax7 = fig.add_subplot(gs[1, 4:6])           # Third column (1 unit)
    ax8 = fig.add_subplot(gs[1, 6:])         # Fourth column (3 units)

    axes = [ax1, ax2, ax3, ax5, ax6, ax7]

    for covariate, ax in zip(covariates_ordered, axes):
        dff = df[df["Covariate"]==covariate]

        sns.set_theme(style="ticks")
        sns.despine()
        #sns.despine(left=True, bottom=True)

        # Plot the points
        sns.stripplot(y='Estimate', x='Method', data=dff, hue="Method", size=8, palette=npg[:n_hues], ax=ax)

        try:
            ax.errorbar(
                y=dff['Estimate'], x=dff['Method'],
                yerr=[dff['err_low'], dff['err_up']], capsize=20, markeredgewidth=10,
                fmt='none', ecolor=colors.values(), linewidth=1.5
            )
        except ValueError:
            # weird error with colors, ignore. but cannot render errobrar caps
            pass

        ref = float(dff[dff["Method"]=="Combined"].loc[:, "Estimate"].values[0])
        ax.axhline(ref, ls="--", color=colors["Combined"])
        ax.set(xlabel=None)
        ax.set_title(covariate, color=grey)
        ax.set_ylabel("Estimate ±95% CI", color=grey)

        if row == rows - 1:
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha='right')
        else:
            ax.xaxis.set_ticks([])


        ax.tick_params(color=grey, labelcolor=grey)
        for spine in ax.spines.values():
            spine.set_edgecolor(grey)

        xlim = ax.get_xlim()
        offset = 0.1*(xlim[1]-xlim[0])
        ax.set_xlim(xlim[0]-offset, xlim[1]+offset)

        # micro tweaks
        if clean_name == "Nurses" and covariate == "age":
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))



    axes2 = [ax4, ax8]

    if use_lolipop:
        lolipop(data=df[df["Method"]!="Combined"], ax=axes2[0], method_order=method_order, covs=covariates_ordered)
    else:
        sns.barplot(data=df[df["Method"]!="Combined"], x="Covariate", y="Error", hue="Method", palette=colors, ax=axes2[0])
        axes2[0].set_ylabel("Absolute Error")

    sns.barplot(data=df, x="Covariate", y="CI_Size", hue="Method", palette=colors, ax=axes2[1])
    axes2[1].set_ylabel("CI Width")
    axes2[1].set_ylim(0, 1.05*np.max(df["CI_Size"]))

    handles, labels = axes2[1].get_legend_handles_labels()
    axes2[1].legend(handles, labels, ncols=2, bbox_to_anchor=(0.1, 1), loc="upper left")
    #axes2[0].legend([], [], framealpha=0)
    axes2[0].legend(ncols=1 if use_lolipop else 5)

    for a in axes2:
        a.set_xlabel(None)

    flataxes = [ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8]
    # for i in range(len(flataxes)):
    #     flataxes[i].annotate(chr(ord('A')+i), xy=(-0.04, 1.1),
    #                xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[0].annotate("A", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[1].annotate("B", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[2].annotate("C", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[3].annotate("G", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[4].annotate("D", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[5].annotate("E", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[6].annotate("F", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[7].annotate("H", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
                        
    #axes[-1][-1].plot(1,1)
    #fig.suptitle(clean_name)
    fig.tight_layout(w_pad=2, h_pad=0)
    fig.savefig(f"figures/collage.{dataname}.pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(f"figures/collage.{dataname}.png", dpi=dpi, bbox_inches="tight")


### Trauma

In [ ]:
use_lolipop = True

if clean_name.startswith("Trauma"):

    from matplotlib.gridspec import GridSpec

    covariates_set = sorted(list({x for x in covariates.flatten() if x is not None}))

    df["Covariate"] = pd.Categorical(df["Covariate"], categories=covariates_set, ordered=True)

    scale = 1.4
    fig = plt.figure(figsize=(scale*12, scale*6))

    # Create a GridSpec with 2 rows and 6 columns (LCM of number of segments)
    gs = GridSpec(2, 12, height_ratios=[1, 1])  # Equal row height for now

    # --- Row 1: 1|1|1|3 split => columns: 1, 1, 1, 3 → total = 6 units
    ax1 = fig.add_subplot(gs[0, 0:2])           # First column (1 unit)
    ax2 = fig.add_subplot(gs[0, 2:4])           # Second column (1 unit)
    ax3 = fig.add_subplot(gs[0, 4:6])           # Third column (1 unit)
    ax4 = fig.add_subplot(gs[0, 6:])         # Fourth column (3 units)

    # --- Row 2: 1.5|1.5|3 => use 6 units as well: 1.5 → 1.5*2 = 3, so 3|3|6
    ax5 = fig.add_subplot(gs[1, 0:3])         # First column (1.5 units)
    ax6 = fig.add_subplot(gs[1, 3:6])         # Second column (1.5 units)
    ax7 = fig.add_subplot(gs[1, 6:])         # Third column (3 units)

    axes = [ax1, ax2, ax3, ax5, ax6]

    for covariate, ax in zip(covariates_set, axes):
        dff = df[df["Covariate"]==covariate]

        sns.set_theme(style="ticks")
        sns.despine()
        #sns.despine(left=True, bottom=True)

        # Plot the points
        sns.stripplot(y='Estimate', x='Method', data=dff, hue="Method", size=8, palette=npg[:n_hues], ax=ax)

        try:
            ax.errorbar(
                y=dff['Estimate'], x=dff['Method'],
                yerr=[dff['err_low'], dff['err_up']], capsize=20, markeredgewidth=10,
                fmt='none', ecolor=colors.values(), linewidth=1.5
            )
        except ValueError:
            # weird error with colors, ignore. but cannot render errobrar caps
            pass

        ref = float(dff[dff["Method"]=="Combined"].loc[:, "Estimate"].values[0])
        ax.axhline(ref, ls="--", color=colors["Combined"])
        ax.set(xlabel=None)
        ax.set_title(covariate, color=grey)
        ax.set_ylabel("Estimate ±95% CI", color=grey)

        if row == rows - 1:
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation=45, ha='right')
        else:
            ax.xaxis.set_ticks([])


        ax.tick_params(color=grey, labelcolor=grey)
        for spine in ax.spines.values():
            spine.set_edgecolor(grey)

        xlim = ax.get_xlim()
        offset = 0.1*(xlim[1]-xlim[0])
        ax.set_xlim(xlim[0]-offset, xlim[1]+offset)

        # micro tweaks
        if clean_name == "Nurses" and covariate == "age":
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))



    axes2 = [ax4, ax7]
    if use_lolipop:
        lolipop(data=df[df["Method"]!="Combined"], ax=axes2[0], method_order=method_order, covs=covariates_set)
    else:
        sns.barplot(data=df[df["Method"]!="Combined"], x="Covariate", y="Error", hue="Method", palette=colors, ax=axes2[0])

    sns.barplot(data=df, x="Covariate", y="CI_Size", hue="Method", palette=colors, ax=axes2[1])
    axes2[1].set_ylabel("CI Width")
    axes2[1].set_ylim(0, 1.05*np.max(df["CI_Size"]))

    handles, labels = axes2[1].get_legend_handles_labels()
    axes2[1].legend(handles, labels, ncols=2)
    #axes2[0].legend([], [], framealpha=0)
    if not use_lolipop:
        axes2[0].legend(ncols=5)

    for a in axes2:
        a.set_xlabel(None)

    flataxes = [ax1, ax2, ax3, ax4, ax5, ax6, ax7]
    # for i in range(len(flataxes)):
    #     flataxes[i].annotate(chr(ord('A')+i), xy=(-0.04, 1.1),
    #                xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[0].annotate("A", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[1].annotate("B", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[2].annotate("C", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[3].annotate("F", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[4].annotate("D", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[5].annotate("E", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    flataxes[6].annotate("G", xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
                        
    #axes[-1][-1].plot(1,1)
    #fig.suptitle(clean_name)
    fig.tight_layout(w_pad=2, h_pad=0)
    fig.savefig(f"figures/collage.{dataname}.pdf", dpi=dpi, bbox_inches="tight")
    fig.savefig(f"figures/collage.{dataname}.png", dpi=dpi, bbox_inches="tight")


## Manhattan

In [ ]:
from scipy import stats

datanames = {"Nurses": "Nurses\n(global intercept)", 
             "Nurses_local_int": "Nurses\n(local intercepts)",
             #"sim-linear-1-out_local_int": "Sim1",
             "sim-nurses-1-out": "Nurses hom\n(global intercept)"
             #"sim-nurses-1-out_local_int": "Nurses hom\n(local intercepts)"
             }
pboxes = list()
for dn, cn in datanames.items():
    pb = pd.read_csv(f"data/summarized/pboxes.{dn}.csv", index_col=0)
    pb["Data"] = cn
    pboxes.append(pb)
pboxes = pd.concat(pboxes)
pboxes.rename(columns={"x": "pbox"}, inplace=True)
pboxes["logp"] = - np.log10(pboxes["pbox"])
pboxes["logpadj"] = -np.log10(stats.false_discovery_control(pboxes["pbox"], method="by"))
#pboxes["logpadj"] = -np.log10(stats.false_discovery_control(pboxes["pbox"], method="bh"))
pboxes["s"] = - np.log2(pboxes["pbox"])

In [ ]:
thresh = 0.05
adjust = False
use_s = True
use_bonferroni = True
val = "logpadj" if adjust else "s" if use_s else "logp"

fig, axes = plt.subplots(1, len(datanames), figsize=(len(datanames)*4, 4))

for i, (ax, dn) in enumerate(zip(axes, datanames.values())):
    pb = pboxes[pboxes["Data"] == dn]
    thresh_adj = thresh
    if use_bonferroni:
        thresh_adj /= len(pb)
    thresh_adj = -np.log2(thresh_adj) if use_s else -np.log10(thresh_adj)
    pb_regular = pb[pb[val] <= thresh_adj]
    pb_outliers = pb[pb[val]>thresh_adj]
    sns.scatterplot(data=pb_regular, x=pb_regular.index, y=val, color=npg[2], ax=ax, zorder=99)
    sns.scatterplot(data=pb_outliers, x=pb_outliers.index, y=val, color=npg[0], ax=ax, zorder=99)

    for i, s in enumerate(pb["s"]):
        ax.plot((i+1,i+1), (0,s), color="lightgrey", zorder=0)

    ax.axhline(thresh_adj, ls="--", color=npg[0], zorder=1)
    ax.axhline(0, color="lightgrey", zorder=0)
    ylabel = r"$-\log_{10}p_{adj.}$" if adjust else r"$-\log_{10}p_{Box}$"
    if use_s: ylabel = r"$s_{Box}$"
    ax.set(xlabel="Center", ylabel=ylabel, title=dn)
    ax.annotate(chr(ord('A')+i), xy=(-0.04, 1.1), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)
    ax.annotate(xy=(0.5,0.75), text=f"Outliers: {len(pb_outliers)}/{len(pb)}",
                xycoords="axes fraction", ha="center", va="center")
fig.tight_layout()
fig.savefig(f"figures/manhattan.png", dpi=dpi, bbox_inches="tight")
fig.savefig(f"figures/manhattan.pdf", dpi=dpi, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6,3))
pb_outliers = pboxes[pboxes["logpadj"] > thresh]
pb_regular = pboxes[pboxes["logpadj"] <= thresh]
sns.stripplot(data=pb_regular, x="logpadj", y="Data", ax=ax, color=npg[1])
sns.stripplot(data=pb_outliers, x="logpadj", y="Data", ax=ax, color=npg[0], zorder=99)
ax.axvline(-np.log10(0.05), ls="--", color=npg[0])
ax.set(ylabel=None, xlabel=r"$-\log_{10}p_{Box}$ (adjusted)")

#fractions = pboxes.groupby("Data")["pbox"].apply(lambda x: (x < 0.05).mean())
pbg = pboxes.groupby("Data")["pbox"]
fractions = pbg.apply(lambda x: (x < 0.05).sum()).astype(str) + "/" + pbg.count().astype(str)

ordered_fractions = fractions.values
ax2 = ax.twinx()
primary_yticks = ax.get_yticks()
ax2.set_yticks(primary_yticks)
ax2.set_yticklabels([f"Outliers: {f}" for f in ordered_fractions])
ax2.set_ylim(ax.get_ylim())

fig.tight_layout()

# Intercepts

In [ ]:
dfi = df_intercept[df_intercept["Method"] != "REML"]
dfi.replace({"Method": {"FE":"Local"}}, inplace=True)
n_sites = len(dfi[dfi["Method"]=="Combined"])
fig, ax = plt.subplots(1,2,figsize=(10,5))
markers = ["o","D","X","p","s","P"]
methods = np.unique(dfi[dfi["Method"]!="Combined"]["Method"])
for i, method in enumerate(methods):
    marker = markers[i]
    ms = 80 * (1-i*0.1)
    c = colors[method if method != "Local" else "FE"]
    err = np.abs((dfi[dfi["Method"]==method].Estimate.values - dfi[dfi["Method"]=="Combined"].Estimate.values))
    sns.boxplot(x=i, y=err, label=method, ax=ax[1], color=c)
    ax[0].scatter(dfi[dfi["Method"]=="Combined"].Estimate, dfi[dfi["Method"]==method].Estimate,
                  label=method, marker=marker, s=ms, color=c)
    
for i in range(len(ax)):
    ax[i].annotate(chr(ord('A')+i), xy=(0.05, 0.95), xycoords="axes fraction", va='center',ha='center', weight="bold", fontsize=14)

ax[0].plot((-1,1), (-1,1), ls="--",color="black", zorder=0)
ax[0].legend()
ax[0].set(xlabel="Intercept (Combined)", ylabel="Intercept (Federated)")

ax[1].set_yscale("log")
ax[1].set_xticklabels(np.array(methods))
ax[1].set(ylabel="Absolute Error")
ax[1].legend([],[], frameon=False)

fig.tight_layout()
fig.savefig(f"figures/intercepts.{dataname}.pdf", dpi=dpi, bbox_inches="tight")

# Cross-validation

In [ ]:
datanames = {#"Nurses": "Nurses\n(global intercept)", 
             #"Nurses_local_int": "Nurses\n(local intercepts)",
             #"sim-linear-1-out_local_int": "Sim1",
             "sim-nurses-1-out": "Nurses linear\n(global intercept)",
             "sim-nurses-1": "Nurses linear\n(global intercept, outlier removed)"
             #"sim-nurses-1-out_local_int": "Nurses linear\n(local intercepts)"
             }


dfs = list()
for dn, cn in datanames.items():
    df = pd.read_csv(f"data/summarized/cross-metrics.{dn}.csv", index_col=0)
    df["Data"] = cn
    dfs.append(df)

df = pd.concat(dfs)

In [ ]:
sns.stripplot(data = df[df["Metric"]=="R2"], x="Data", y="Value", hue="Data")
sns.boxplot(data = df[df["Metric"]=="R2"], x="Data", y="Value", hue="Data", fill=None)
plt.ylabel(r"$R^2$")
plt.xlabel(None)

# Reproducibility

In [ ]:
import sys
import platform
import importlib.metadata as metadata

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("\nInstalled packages:")

for dist in sorted(metadata.distributions(), key=lambda d: d.metadata["Name"].lower()):
    name = dist.metadata["Name"]
    version = dist.version
    print(f"  {name}=={version}")